# Best encoding settings

The aim here is to train a model on existing data in order to estimate the FFmpeg parameters that optimise any arbitrary criterion.

In [1]:
import pathlib
import subprocess
import tempfile

import cutcutcodec

from mendevi.encode import get_transcode_cmd
from mendevi.models import Solver
from mendevi.models.nn import EnergyNN, QualityNN

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Train the model

Before making predictions, you first need to train a model.

### Quality model

Quality does not depend on the machine. We therefore produce just one large, unique model.

In [3]:
# declaration
quality_model = QualityNN(
    output_labels=["log_size_per_frame", "psnr", "ssim"],
    weights="quality",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=15000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=50_000,  # number of model parameters, default based on number of observations
)

In [4]:
# train (on cpu encoders only)
if not quality_model.is_fit():
    quality_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 17853 observations
█ █ Save model into /home/robin/.cache/mendevi/quality_29_3_50000.pk.xz...                                            
█ █ █ 31.96 ms
█ █ 8 min 44 s
█ 9 min 18 s
Save model into /home/robin/.cache/mendevi/quality.pk.xz...
█ 33.68 ms


In [5]:
# validation
quality_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 4551 observations
█ █ 62.12 ms
█ 15.01 s
MAPE of log_size_per_frame: 2.21%
RMS  of log_size_per_frame: 0.1174
MAPE of psnr: 2.91%
RMS  of psnr: 1.889
MAPE of ssim: 1.60%
RMS  of ssim: 0.0241
MAPE of size_per_frame: 20.66%
RMS  of size_per_frame: 6.453e+04


### Energy model

Energy depends ont the machine. We therefore produce several models, one per hostname.

In [6]:
energy_model = EnergyNN(
    weights="energy",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=8000,  # number of training iterations, default 10000
    learning_rate=1e-4,  # adam gradient decay learning rate, default 1e-4
    total=2000,  # number of model parameters, default based on number of observations
)

In [8]:
if not energy_model.is_fit():
    energy_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 6500 observations
█ █ Save model into /home/robin/.cache/mendevi/energy_29_2_2000.pk.xz...                                              
█ █ █ 4.68 ms
█ █ 19.82 s
█ Fit on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 2539 observations
█ █ Load model from /home/robin/.cache/mendevi/energy_29_2_2000.pk.xz...
█ █ █ 3.55 ms
█ █ Save model into /home/robin/.cache/mendevi/energy_29_2_2000.pk.xz...                                              
█ █ █ 4.64 ms
█ █ 12.75 s
█ Fit on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 850 observations
█ █ Load model from /home/robin/.cache/mendevi/energy_29_2_2000.pk.xz...
█ █ █ 2.20 ms
█ █ Save model into /home/robin/.cache/mendevi/energy_29_2_2000.pk.xz...                                              
█ █ █ 4.74

In [9]:
# validation
energy_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 1687 observations
█ █ 20.44 ms
█ Validate on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 676 observations
█ █ 10.76 ms
█ Validate on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 256 observations
█ █ 6.24 ms
█ Validate on cluster hostname=nova-12.lyon.grid5000.fr...
█ █ this cluster contains 476 observations
█ █ 8.50 ms
█ Validate on cluster hostname=paradoxe-32.rennes.grid5000.fr...
█ █ this cluster contains 618 observations
█ █ 9.81 ms
█ Validate on cluster hostname=suzanne...
█ █ this cluster contains 405 observations
█ █ 8.09 ms
█ Validate on cluster hostname=yeti-2.grenoble.grid5000.fr...
█ █ this cluster contains 433 observations
█ █ 8.35 ms
█ 33.16 s
MAPE of log_act_duration_per_frame: 22.02%
RMS  of log_act_duration_per_frame:

## Use the model to perform predictions

The idea is to run a huge number of predictions using the trained model in order to explore a very large range of scenarios.

In [11]:
# example best compression for 40 dB

def best_dbrate(**kwd: dict[str]) -> float:
    """Loss function of debit rate distortion around 35 dB."""
    db_loss = 0.0 if 39 < kwd["psnr"] < 41 else ((kwd["psnr"]-40)**2)-1
    rate_loss = kwd["log_size_per_frame"]
    return db_loss + 1e-2*rate_loss

solver = Solver(quality_model, best_dbrate)
tested_combinations, loss, cmd = solver.solve(
    video,
    encoder=["libx264", "libx265", "libaom-av1", "libsvtav1", "libvvenc"],
    return_cmd=True,
)

Solve...
█ Predict 'Quality estimation based on a neuronal network'...
█ █ check args
█ █ vectorize args
█ █ predict
█ █ Predict for cluster ...
█ █ █ this cluster contains 990 observations
█ █ █ 12.51 ms
█ █ 20.50 ms
█ Result for loss = 0.02904:
█ █ effort            : medium
█ █ encoder           : libvvenc
█ █ height            : 720
█ █ log_size_per_frame: 2.904
█ █ mode              : crf
█ █ nb_frames         : 294
█ █ psnr              : 34.74
█ █ quality           : 0.6471
█ █ rms_sobel         : 0.1544
█ █ rms_time_diff     : 0.08223
█ █ spatial_dct       : 0.05345
█ █ ssim              : 0.8735
█ █ temporal_dct      : 0.01171
█ █ width             : 1280
  0%|                                                                                          | 0/2.0 [00:00<?, ?s/s]try with 'sudo /home/robin/.pyenv/versions/3.14.2/envs/3.14/bin/python3'
or give rights with 'sudo sysctl -w kernel.perf_event_paranoid=0'
or create /etc/sysctl.d/99-perf.conf with 'kernel.perf_event_paranoid 

In [12]:
# perform the conversion
out_video = pathlib.Path(tempfile.gettempdir()) / "video_test.mp4"
cmd.output = [*cmd.output[:-1], str(out_video)]
output, activity = cmd.run()

  0%|                                                                                         | 0/9.81 [00:00<?, ?s/s]try with 'sudo /home/robin/.pyenv/versions/3.14.2/envs/3.14/bin/python3'
or give rights with 'sudo sysctl -w kernel.perf_event_paranoid=0'
or create /etc/sysctl.d/99-perf.conf with 'kernel.perf_event_paranoid = 0'
Error:
No supported events found.
Access to performance monitoring and observability operations is limited.
Consider adjusting /proc/sys/kernel/perf_event_paranoid setting to open
access to performance monitoring and observability operations for processes
without CAP_PERFMON, CAP_SYS_PTRACE or CAP_SYS_ADMIN Linux capability.
More information can be found at 'Perf events and tool security' document:
https://www.kernel.org/doc/html/latest/admin-guide/perf-security.html
perf_event_paranoid setting is 3:
  -1: Allow use of (almost) all events by all users
      Ignore mlock limit after perf_event_mlock_kb without CAP_IPC_LOCK
>= 0: Disallow raw and ftrace function

POC    31 TId: 0 (IDR_W_RADL, I-SLICE, QP 25, TF 2)      18392 bits [Y 53.6026 dB    U 61.9857 dB    V 58.6485 dB] [ET     0 ] [L0 ] [L1 ]
POC    15 TId: 1 (      RADL, B-SLICE, QP 30, TF 1)       2144 bits [Y 59.5417 dB    U 73.9381 dB    V 67.4565 dB] [ET     0 ] [L0 31 ] [L1 31 ]
POC     7 TId: 2 (      RADL, B-SLICE, QP 33, TF 0)        680 bits [Y 60.4770 dB    U 73.4433 dB    V 68.6529 dB] [ET     0 ] [L0 15 31 ] [L1 15 31 ]
POC     3 TId: 3 (      RADL, B-SLICE, QP 35)              304 bits [Y 60.5525 dB    U 72.9475 dB    V 68.0413 dB] [ET     0 ] [L0 7 15 ] [L1 7 15 ]
POC     1 TId: 4 (      RADL, B-SLICE, QP 37)              296 bits [Y 61.9520 dB    U 72.3059 dB    V 67.7520 dB] [ET     0 ] [L0 3 7 ] [L1 3 7 ]
POC     0 TId: 5 (      RADL, B-SLICE, QP 38)              256 bits [Y 61.8858 dB    U 72.0105 dB    V 67.5561 dB] [ET     0 ] [L0 1 3 ] [L1 1 3 ]
POC     2 TId: 5 (      RADL, B-SLICE, QP 38)              256 bits [Y 59.7888 dB    U 73.9381 dB    V 66.9169 dB] [ET    

  1%|▊                                                                              | 0.1/9.81 [00:02<04:12, 25.95s/s]

POC    22 TId: 5 (      RADL, B-SLICE, QP 38)              352 bits [Y 54.3346 dB    U 73.8669 dB    V 62.7541 dB] [ET     0 ] [L0 21 19 ] [L1 23 31 ]
POC    27 TId: 3 (      RADL, B-SLICE, QP 35)             2120 bits [Y 51.9393 dB    U 62.7009 dB    V 57.1312 dB] [ET     0 ] [L0 23 15 ] [L1 31 23 ]
POC    25 TId: 4 (      RADL, B-SLICE, QP 38)              648 bits [Y 51.8565 dB    U 63.4878 dB    V 58.1958 dB] [ET     0 ] [L0 23 15 ] [L1 27 31 ]
POC    24 TId: 5 (      RADL, B-SLICE, QP 38)              776 bits [Y 51.4333 dB    U 63.5248 dB    V 58.7064 dB] [ET     0 ] [L0 23 15 ] [L1 25 27 ]
POC    26 TId: 5 (      RADL, B-SLICE, QP 38)              528 bits [Y 51.7268 dB    U 61.8050 dB    V 57.5961 dB] [ET     0 ] [L0 25 23 ] [L1 27 31 ]
POC    29 TId: 4 (      RADL, B-SLICE, QP 38)             1176 bits [Y 49.7301 dB    U 60.4703 dB    V 55.4157 dB] [ET     0 ] [L0 27 15 ] [L1 31 27 ]


  5%|██▉                                                            | 0.45999999999999996/9.81 [00:03<01:02,  6.73s/s]

POC    28 TId: 5 (      RADL, B-SLICE, QP 38)              920 bits [Y 50.0655 dB    U 59.8985 dB    V 55.7261 dB] [ET     0 ] [L0 27 15 ] [L1 29 31 ]


... (93 outputs left out)

POC   278 TId: 5 (      RASL, B-SLICE, QP 50)             2728 bits [Y 37.1793 dB    U 40.6294 dB    V 42.2299 dB] [ET     0 ] [L0 277 275 ] [L1 279 287 ]
POC   283 TId: 3 (      RASL, B-SLICE, QP 44)             5616 bits [Y 40.4871 dB    U 42.3382 dB    V 43.7797 dB] [ET     1 ] [L0 279 271 ] [L1 287 279 ]
POC   281 TId: 4 (      RASL, B-SLICE, QP 47)             3360 bits [Y 38.7072 dB    U 41.9414 dB    V 43.0790 dB] [ET     0 ] [L0 279 271 ] [L1 283 287 ]
POC   280 TId: 5 (      RASL, B-SLICE, QP 49)             2872 bits [Y 38.5649 dB    U 41.3457 dB    V 42.4413 dB] [ET     0 ] [L0 279 271 ] [L1 281 283 ]
POC   282 TId: 5 (      RASL, B-SLICE, QP 46)             3560 bits [Y 39.4436 dB    U 42.4589 dB    V 43.3932 dB] [ET     0 ] [L0 281 279 ] [L1 283 287 ]
POC   285 TId: 4 (      RASL, B-SLICE, QP 42)             4464 bits [Y 41.3834 dB    U 43.6041 dB    V 44.4062 dB] [ET     0 ] [L0 283 271 ] [L1 287 283 ]


 93%|████████████████████████████████████████████████████████████████████████▋     | 9.14/9.81 [01:16<00:05,  8.39s/s]

POC   284 TId: 5 (      RASL, B-SLICE, QP 43)             2928 bits [Y 41.0151 dB    U 43.5780 dB    V 44.9189 dB] [ET     0 ] [L0 283 271 ] [L1 285 287 ]
POC   286 TId: 5 (      RASL, B-SLICE, QP 47)             1672 bits [Y 41.1986 dB    U 42.5291 dB    V 44.6196 dB] [ET     0 ] [L0 285 283 ] [L1 287 285 ]
POC   291 TId: 3 (      STSA, B-SLICE, QP 38)             1640 bits [Y 52.7746 dB    U 49.0131 dB    V 49.7361 dB] [ET     0 ] [L0 287 ] [L1 287 ]
POC   289 TId: 4 (      STSA, B-SLICE, QP 40)             3080 bits [Y 47.1642 dB    U 46.6275 dB    V 48.2252 dB] [ET     0 ] [L0 287 291 ] [L1 291 287 ]


POC   288 TId: 5 (      STSA, B-SLICE, QP 43)             1840 bits [Y 44.9629 dB    U 44.1652 dB    V 47.0046 dB] [ET     0 ] [L0 287 289 ] [L1 289 291 ]
POC   290 TId: 5 (      STSA, B-SLICE, QP 42)             2168 bits [Y 47.7114 dB    U 45.5077 dB    V 48.0235 dB] [ET     0 ] [L0 289 287 ] [L1 291 289 ]
POC   293 TId: 4 (      STSA, B-SLICE, QP 37)              512 bits [Y 70.4282 dB    U 53.2066 dB    V 51.9959 dB] [ET     0 ] [L0 291 287 ] [L1 291 287 ]
POC   292 TId: 5 (      STSA, B-SLICE, QP 39)              672 bits [Y 55.8425 dB    U 51.6178 dB    V 51.8603 dB] [ET     0 ] [L0 291 287 ] [L1 293 291 ]

vvenc [info]:	Total Frames |   Bitrate     Y-PSNR    U-PSNR    V-PSNR    YUV-PSNR   
vvenc [info]:	      294    a     171.8176   39.5734   47.6739   46.6522   38.4774


In [13]:
# final verification
out = subprocess.run(
    ["mendevi", "probe", str(out_video), "-r", str(video), "-d", str(out_video.with_suffix(".db"))],
    check=True,
)

Parse configuration...
█ video     : /tmp/video_test.mp4
█ database  : /tmp/video_test.db (just created)
█ ref       : intro.webm
█ lpips_alex: no
█ lpips_vgg : no
█ psnr      : yes
█ rms_sobel : yes
█ rms_time_diff: yes
█ spatial_dct: yes
█ ssim      : yes
█ temporal_dct: yes
█ uvq       : no
█ vif       : no
█ vmaf      : no
Probe 1/2...                                                     
█ Get properties of intro.webm...
█ █ eotf         : bt1361e, bt1361
█ █ range        : tv
█ █ duration     : 9.81 s
█ █ width        : 1280 pixels
█ █ name         : intro.webm
█ █ height       : 720 pixels
█ █ pix fmt      : yuv420p
█ █ gamut        : bt709
█ █ codec        : vp9
█ █ frames       : 294 frames
█ █ fps          : 29.97 Hz
█ █ size         : 0.50 MB
█ █ rms_sobel    : 0.15 +/- 0.07                                                                                                                                                 
█ █ rms_time_diff: 0.08 +/- 0.05
█ █ spatial_dct  : 0.05 +/